# E1 — Baseline preditivo de propensão (v0)

Issue #27. Modelo supervisionado simples que estima a **propensão de conversão**
(`P(y = "yes")`, assinar o depósito a prazo) sobre a `modeling_table` (sem `duration`).

**Dois usos:**
1. **Feature de contexto para o bandit (E3):** o score de propensão entra como sinal
   engenheirado no contexto do cliente.
2. **Baseline preditivo:** representa a abordagem "ML clássico" (prever quem assina),
   contraste da plataforma adaptativa.

**Decisões de modelagem:**
- **Sem rebalanceamento** (nada de SMOTE/under/oversample). O score precisa ser uma
  probabilidade ~calibrada; resampling distorce a taxa base (~11% positivos) e infla a
  propensão. Mantemos a distribuição natural.
- **Métrica: ROC-AUC + PR-AUC** (threshold-free). Acurácia é enganosa em base desbalanceada.
- **Score para a E3 gerado out-of-fold** (`cross_val_predict`), para que a propensão de
  cada cliente venha de um modelo que não o viu no treino — evita leakage a jusante.
- **Atalho LGBM:** sugerido na issue como alternativa; fica para um v1. Logística basta no v0.

In [1]:
import logging

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from bankmarketing.data import MODELING_TABLE_PATH, PROCESSED_DIR, load_modeling_table

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger(__name__)
pd.set_option("display.max_columns", None)

SEED = 42
TARGET = "y"

In [2]:
modeling_table = load_modeling_table(MODELING_TABLE_PATH)
assert "duration" not in modeling_table.columns, "leakage: duration não pode estar presente"

# Target binário: yes -> 1, no -> 0
y = (modeling_table[TARGET] == "yes").astype(int)
X = modeling_table.drop(columns=[TARGET])

print(f"linhas: {len(X):,} | features: {X.shape[1]}")
print(f"prevalência de conversão (y=yes): {y.mean():.3%}  ({y.sum():,} de {len(y):,})")

INFO Modeling table carregada de /home/gabriemello/drive/3fiap/FASE 5 - MLOPS/BankMarketing/data/processed/modeling_table.parquet (41188 linhas x 21 colunas) — source=henriqueyamahata/bank-marketing, version=with social/economic context, license=CC BY 4.0, leakage_removed=duration


linhas: 41,188 | features: 20
prevalência de conversão (y=yes): 11.265%  (4,640 de 41,188)


In [3]:
# Tipagem das colunas: categóricas (object/string) recebem one-hot; numéricas/bool são escaladas.
cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()

print(f"categóricas ({len(cat_cols)}): {cat_cols}")
print(f"numéricas  ({len(num_cols)}): {num_cols}")

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ]
)

# Sem class_weight: priorizamos calibração do score (uso como feature de contexto na E3).
model = Pipeline(
    steps=[
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
    ]
)

categóricas (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
numéricas  (10): ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'foi_contatado_antes']


## Avaliação honesta — split estratificado treino/teste

Métricas reportadas no conjunto de teste, mantido na distribuição natural (~11%).

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

model.fit(X_train, y_train)
proba_test = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)
baseline_pr = y_test.mean()  # PR-AUC de um classificador aleatório = prevalência

print(f"ROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}  (baseline aleatório = {baseline_pr:.4f})")
print(f"lift PR-AUC sobre o aleatório: {pr_auc / baseline_pr:.2f}x")

ROC-AUC : 0.8009
PR-AUC  : 0.4652  (baseline aleatório = 0.1126)
lift PR-AUC sobre o aleatório: 4.13x


In [5]:
# Relatório no threshold padrão 0.5 — apenas referência; a decisão real não usa threshold fixo.
pred_test = (proba_test >= 0.5).astype(int)
print(classification_report(y_test, pred_test, target_names=["no", "yes"]))

              precision    recall  f1-score   support

          no       0.91      0.99      0.95      7310
         yes       0.69      0.22      0.33       928

    accuracy                           0.90      8238
   macro avg       0.80      0.60      0.64      8238
weighted avg       0.88      0.90      0.88      8238



## Score de propensão para a E3 (out-of-fold, sem leakage)

Geramos a propensão de **todos** os clientes via `cross_val_predict`: o score de cada
linha vem de um fold que não a viu no treino. Esse parquet é o insumo de contexto para o
bandit (E3) e fica em `data/processed/propensity_scores.parquet`.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
propensity_oof = cross_val_predict(
    model, X, y, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

oof_roc = roc_auc_score(y, propensity_oof)
oof_pr = average_precision_score(y, propensity_oof)
print(f"OOF ROC-AUC: {oof_roc:.4f} | OOF PR-AUC: {oof_pr:.4f}")

scores = pd.DataFrame(
    {"propensity_score": propensity_oof},
    index=modeling_table.index,
)
scores.index.name = "client_idx"
scores["propensity_score"].describe()

OOF ROC-AUC: 0.7915 | OOF PR-AUC: 0.4514


count    41188.000000
mean         0.112670
std          0.147959
min          0.001929
25%          0.039907
50%          0.056886
75%          0.093715
max          0.906451
Name: propensity_score, dtype: float64

In [7]:
out_path = PROCESSED_DIR / "propensity_scores.parquet"
scores.to_parquet(out_path)
print(f"score de propensão salvo em: {out_path}")
print("alinhado por posição com modeling_table (client_idx = índice da linha).")

score de propensão salvo em: /home/gabriemello/drive/3fiap/FASE 5 - MLOPS/BankMarketing/data/processed/propensity_scores.parquet
alinhado por posição com modeling_table (client_idx = índice da linha).


## Notas (DoD da issue #27)

- [x] Modelo treinado sobre `modeling_table.parquet` (sem `duration`).
- [x] Métrica reportada (ROC-AUC + PR-AUC) no notebook.
- [x] Score de propensão disponível como feature de contexto para E3
      (`data/processed/propensity_scores.parquet`, gerado out-of-fold).
- [x] Sem leakage: `duration` ausente (assert acima); score OOF evita treino-no-próprio-dado.

**Limitações (v0):** logística linear; sem tuning de hiperparâmetros; sem calibração
explícita (`CalibratedClassifierCV`) — avaliar em v1 junto de LightGBM se a E3 precisar
de propensão mais calibrada/discriminativa.